# 04 — Fine-Tuning con Imágenes SEMIC

Transfer learning usando el modelo entrenado con datasets públicos y ajustándolo con imágenes reales de SEMIC.

**Estrategia**:
1. Cargar el mejor modelo de la fase anterior
2. Congelar las primeras capas del backbone
3. Entrenar con learning rate bajo sobre el dataset SEMIC
4. Evaluar mejora en imágenes reales

In [ ]:
# === Setup para Google Colab ===
import os

if os.getenv('COLAB_RELEASE_TAG'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/neurodrive
    !pip install -q ultralytics
    print('✅ Entorno Colab configurado')
else:
    print('✅ Entorno local detectado')

In [ ]:
# === Configuración del Fine-Tuning ===

# Modelo pre-entrenado (de la fase anterior)
PRETRAINED_MODEL = 'results/baseline_v1/weights/best.pt'  # Cambiar por el mejor modelo

# Dataset SEMIC
SEMIC_YAML = 'configs/datasets/semic.yaml'  # Actualizar cuando el dataset esté listo

# Hiperparámetros de fine-tuning
EPOCHS = 30
BATCH_SIZE = 8        # Batch pequeño para dataset pequeño
IMG_SIZE = 640
LR = 0.001            # Learning rate bajo para fine-tuning
FREEZE_LAYERS = 10    # Congelar primeras N capas
PATIENCE = 10
PROJECT = 'results'
RUN_NAME = 'finetune_semic_v1'

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from ultralytics import YOLO
import torch

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

## Verificar Dataset SEMIC

Antes de entrenar, verificamos que el dataset SEMIC esté disponible y correctamente anotado.

In [ ]:
semic_dir = Path('data/semic')

if not any(semic_dir.glob('**/*.jpg')) and not any(semic_dir.glob('**/*.png')):
    print('⚠️ No se encontraron imágenes en data/semic/')
    print('\nPasos para preparar el dataset SEMIC:')
    print('1. Colocar las imágenes en data/semic/images/')
    print('2. Anotar con herramienta como LabelImg o Roboflow')
    print('3. Colocar labels YOLO en data/semic/labels/')
    print('4. Actualizar configs/datasets/semic.yaml con las clases')
    print('5. Generar el data.yaml para Ultralytics')
else:
    from src.data.validate import validate_dataset
    
    images_dir = semic_dir / 'images'
    labels_dir = semic_dir / 'labels'
    
    if images_dir.exists() and labels_dir.exists():
        stats = validate_dataset(str(images_dir), str(labels_dir))
    else:
        print('⚠️ Estructura esperada: data/semic/images/ y data/semic/labels/')

## Cargar Modelo Pre-entrenado

Cargamos el modelo que ya fue entrenado con los datasets públicos.

In [ ]:
if Path(PRETRAINED_MODEL).exists():
    model = YOLO(PRETRAINED_MODEL)
    print(f'✅ Modelo cargado: {PRETRAINED_MODEL}')
    print(f'   Parámetros: {sum(p.numel() for p in model.model.parameters()):,}')
else:
    print(f'⚠️ Modelo no encontrado: {PRETRAINED_MODEL}')
    print('Usando modelo pretrained de Ultralytics como respaldo')
    model = YOLO('yolo11s.pt')

## Fine-Tuning

Entrenamos con learning rate bajo y capas congeladas para adaptar el modelo a las imágenes de SEMIC sin perder lo aprendido.

In [ ]:
# Fine-tuning
# NOTA: Descomentar cuando el dataset SEMIC esté listo

# results = model.train(
#     data=SEMIC_YAML,
#     epochs=EPOCHS,
#     batch=BATCH_SIZE,
#     imgsz=IMG_SIZE,
#     lr0=LR,
#     freeze=FREEZE_LAYERS,
#     patience=PATIENCE,
#     project=PROJECT,
#     name=RUN_NAME,
#     plots=True,
# )

print('⚠️ Fine-tuning comentado — descomentar cuando el dataset SEMIC esté listo')

## Comparar Antes y Después

Comparamos las métricas del modelo original vs el fine-tuned en imágenes SEMIC.

> **Output esperado**: Tabla comparativa mostrando mejora en métricas sobre imágenes reales.

In [ ]:
# Comparar métricas (descomentar cuando se tenga el modelo fine-tuned)

# from src.models.evaluate import evaluate_model, compare_models
# 
# finetuned_model = f'{PROJECT}/{RUN_NAME}/weights/best.pt'
# 
# if Path(finetuned_model).exists():
#     df = compare_models(
#         [PRETRAINED_MODEL, finetuned_model],
#         SEMIC_YAML,
#         output_dir='results/evaluacion_semic'
#     )
#     display(df)

print('⚠️ Comparativa pendiente hasta completar fine-tuning')

## Notas

- Si el dataset SEMIC es muy pequeño (<100 imágenes), considerar usar augmentation agresiva
- Si hay clases nuevas en SEMIC, puede ser necesario re-entrenar las capas finales
- Documentar las métricas finales para el reporte de InnovaTecNM